In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

In [0]:
# ==================== Load Source Tables ====================

exposure = (
    spark.sql(f"SELECT * FROM `p-cdh`.primary.get_all_exposures(DATE'{run_date}')")
    .filter(col("PrimaryPartyFlag") == True)  
)

dexposure_closing = (
    spark.sql(f"SELECT ExposureKey AS AssetKey, CAST(ClosedDate AS date) AS ClosingDate FROM `{env}-cdh`.derivative.get_all_exposures_by_business_date('{run_date}')")
    .dropDuplicates(["AssetKey"])
)

portfolio_tbl = f"`{env}-cdh`.primary.portfolio"
_max_portfolio_date = spark.table(portfolio_tbl).agg(F.max("BusinessDate")).first()[0]
portfolio = (
    spark.table(portfolio_tbl)
    .filter(F.col("BusinessDate") == _max_portfolio_date)
    .select("PortfolioId", "PortfolioName")
)

# ==================== Build AllAccounts ====================

allaccounts_df = (
    exposure.select('PortfolioName', 'ExposureKey', 'SourceSystem')
    .join(portfolio, on="PortfolioName", how="inner")
    .withColumn("PortfolioId",
        F.when(col("SourceSystem") == "qca", F.lit(31))
         .when(col("SourceSystem") == "arotron", F.lit(32))
         .otherwise(col("PortfolioId"))
    )
    .select(
        col("PortfolioId"),
        col("ExposureKey").alias("AssetKey"),
        run_date_lit.alias("datadate")
    )
    .dropDuplicates(["AssetKey"])
    .join(dexposure_closing, on="AssetKey", how="left")
    .select("PortfolioId", "AssetKey", "ClosingDate", "datadate")
)

print(f"dtransfer_allaccounts rows: {allaccounts_df.count():,}")

In [0]:
# ==================== Write dtransfer_allaccounts ====================

target_allaccounts = f"`{env}-cdh-delivery`.it.dtransfer_allaccounts"

(
    allaccounts_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_allaccounts)
)

print(f"Written to {target_allaccounts}")